# Step 4 — Structural Alignment + Fragment Analysis (Corrected)

**Corrected version** incorporating all fixes:
- `gemmi.one_letter_code([res.name for res in polymer])` (API change fix)
- Removed `model.name` reference (uses `st[0]` directly)
- Sliding-window sequence matching for postfusion S2 chain extraction
- Fragment analysis with artifact removal (TM > 1.0 overflow, alnlen < 15)

**Reads:** `structure_sources.json` (Step 2), `filled_structures.json` (Step 3), `panel.json` (Step 1)
**Produces:** `alignment_results.json` with whole-domain and fragment-level results

In [ ]:
import subprocess, sys, os, importlib, json, time

for pkg in ("requests", "gemmi", "numpy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import gemmi
import numpy as np
print("gemmi:", gemmi.__version__)

In [ ]:
# Install Foldseek
FOLDSEEK_DIR = "/content/foldseek"
FOLDSEEK_BIN = os.path.join(FOLDSEEK_DIR, "bin", "foldseek")

if not os.path.exists(FOLDSEEK_BIN):
    print("Downloading Foldseek...")
    os.system("wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz -O /tmp/foldseek.tar.gz")
    os.system("tar xzf /tmp/foldseek.tar.gz -C /content/")
    os.system("rm /tmp/foldseek.tar.gz")

print("Foldseek:", subprocess.run([FOLDSEEK_BIN, "version"], capture_output=True, text=True).stdout.strip())

In [ ]:
MOUNT_DRIVE = True
BASE_DIR = "/content"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP1_DIR = os.path.join(BASE_DIR, "step1")
STEP2_DIR = os.path.join(BASE_DIR, "step2")
STEP3_DIR = os.path.join(BASE_DIR, "step3")
STEP4_DIR = os.path.join(BASE_DIR, "step4")
os.makedirs(STEP4_DIR, exist_ok=True)

with open(os.path.join(STEP1_DIR, "panel.json")) as f:
    panel = json.load(f)
with open(os.path.join(STEP2_DIR, "structure_sources.json")) as f:
    structure_sources = json.load(f)
with open(os.path.join(STEP3_DIR, "filled_structures.json")) as f:
    filled_structures = json.load(f)

print("Loaded all prior step outputs.")

## Chain extraction (corrected for gemmi API + postfusion S2)

In [ ]:
CHAINS_DIR = os.path.join(STEP4_DIR, "chains")
os.makedirs(CHAINS_DIR, exist_ok=True)


def extract_chain_by_sequence_match(cif_path, target_seq_fragment, output_pdb_path, min_match_len=50):
    doc = gemmi.cif.read(cif_path)
    st = gemmi.make_structure_from_block(doc[0])
    st.setup_entities()

    best_chain = None
    best_overlap = 0

    chain_sequences = []
    for model in st:
        for chain in model:
            polymer = chain.get_polymer()
            if not polymer:
                continue
            chain_seq = gemmi.one_letter_code([res.name for res in polymer])
            chain_sequences.append((chain.name, chain_seq))
        break

    target_seq = target_seq_fragment.upper()
    windows_to_try = list(range(0, max(1, len(target_seq) - min_match_len), min_match_len))

    for window_start in windows_to_try:
        fragment = target_seq[window_start:window_start + min_match_len]
        if len(fragment) < 20:
            continue
        for chain_name, chain_seq in chain_sequences:
            overlap = 0
            for i in range(len(fragment)):
                if fragment[i:] in chain_seq:
                    overlap = len(fragment) - i
                    break
            if overlap > best_overlap:
                best_overlap = overlap
                best_chain = chain_name
        if best_overlap >= min_match_len:
            break

    if best_chain is None or best_overlap < 20:
        print(f"  WARNING: no good chain match in {cif_path} (overlap={best_overlap})")
        return None

    new_st = gemmi.Structure()
    new_st.cell = st.cell
    new_model = gemmi.Model("1")
    for chain in st[0]:
        if chain.name == best_chain:
            new_model.add_chain(chain)
            break
    new_st.add_model(new_model)
    new_st.write_pdb(output_pdb_path)
    return best_chain


# Spike chains
spike_chain_files = {}
for state, meta in structure_sources["spike"]["selected"].items():
    cif_path = meta["file_path"]
    out_path = os.path.join(CHAINS_DIR, f"spike_{state}_{meta['pdb_id']}_chainA.pdb")
    spike_seq = panel["entries"]["spike"]["sequence"]
    chain_id = extract_chain_by_sequence_match(cif_path, spike_seq, out_path)
    if chain_id is not None and os.path.exists(out_path):
        spike_chain_files[state] = out_path
        print(f"  spike {state}: chain {chain_id} from {meta['pdb_id']}")
    else:
        print(f"  spike {state}: FAILED -- excluded")

# Cardiac antigen chains
antigen_chain_files = {}
CARDIAC_ANTIGENS = ["MYH6", "TPM1", "TPM3", "ACTC1"]

for symbol in CARDIAC_ANTIGENS:
    entry = structure_sources["cardiac_antigens"][symbol]
    antigen_seq = panel["entries"][symbol]["sequence"]
    if entry["best_pdb_id"] and entry["coverage_pct"] and entry["coverage_pct"] >= 90:
        cif_path = entry["file_path"]
        source_label = f"experimental ({entry['best_pdb_id']})"
    else:
        regions = filled_structures["cardiac_antigens"][symbol]["regions"]
        if regions:
            cif_path = regions[0]["file_path"]
            source_label = f"alphafold_db"
        else:
            print(f"  {symbol}: ERROR -- no structure available")
            continue
    out_path = os.path.join(CHAINS_DIR, f"{symbol}_chain.pdb")
    chain_id = extract_chain_by_sequence_match(cif_path, antigen_seq, out_path)
    if chain_id is not None and os.path.exists(out_path):
        antigen_chain_files[symbol] = out_path
        print(f"  {symbol}: chain {chain_id} from {source_label}")
    else:
        print(f"  {symbol}: FAILED -- excluded")

print(f"\nReady: {len(spike_chain_files)} states x {len(antigen_chain_files)} antigens = "
      f"{len(spike_chain_files) * len(antigen_chain_files)} comparisons")

## Whole-domain Foldseek alignment

In [ ]:
RESULTS_DIR = os.path.join(STEP4_DIR, "foldseek_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
TMP_DIR = "/tmp/foldseek_tmp"
os.makedirs(TMP_DIR, exist_ok=True)

OUTPUT_COLS = "query,target,alntmscore,fident,alnlen,qlen,tlen,evalue,lddt"
all_results = []

for state, spike_pdb in spike_chain_files.items():
    for symbol, antigen_pdb in antigen_chain_files.items():
        label = f"{state}_vs_{symbol}"
        out_file = os.path.join(RESULTS_DIR, f"{label}.m8")
        cmd = [FOLDSEEK_BIN, "easy-search", spike_pdb, antigen_pdb, out_file, TMP_DIR,
               "--format-output", OUTPUT_COLS, "--exhaustive-search", "1", "-e", "10"]
        subprocess.run(cmd, capture_output=True, text=True)

        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            with open(out_file) as f:
                for line in f:
                    fields = line.strip().split("\t")
                    if len(fields) >= 9:
                        hit = {"spike_state": state, "spike_pdb": os.path.basename(spike_pdb),
                               "antigen": symbol, "antigen_pdb": os.path.basename(antigen_pdb),
                               "tm_score": float(fields[2]), "seq_identity": float(fields[3]),
                               "aln_length": int(fields[4]), "query_length": int(fields[5]),
                               "target_length": int(fields[6]), "evalue": float(fields[7]),
                               "lddt": float(fields[8])}
                        all_results.append(hit)
                        print(f"  {label}: TM={hit['tm_score']:.4f}  E={hit['evalue']:.2e}")
        else:
            all_results.append({"spike_state": state, "spike_pdb": os.path.basename(spike_pdb),
                                "antigen": symbol, "antigen_pdb": os.path.basename(antigen_pdb),
                                "tm_score": None, "seq_identity": None, "aln_length": None,
                                "query_length": None, "target_length": None, "evalue": None,
                                "lddt": None, "note": "no Foldseek hit returned"})
            print(f"  {label}: no hit")
        time.sleep(0.2)

print(f"\n{len(all_results)} pairwise results collected.")

## Fragment-based local patch analysis (with artifact removal)

In [ ]:
FRAG_DIR = os.path.join(STEP4_DIR, "fragments")
SPIKE_FRAG_DIR = os.path.join(FRAG_DIR, "spike")
ANTIGEN_FRAG_DIR = os.path.join(FRAG_DIR, "antigens")
FRAG_RESULTS_DIR = os.path.join(STEP4_DIR, "fragment_results")
FRAG_TMP_DIR = "/tmp/foldseek_frag_tmp"
for d in (SPIKE_FRAG_DIR, ANTIGEN_FRAG_DIR, FRAG_RESULTS_DIR, FRAG_TMP_DIR):
    os.makedirs(d, exist_ok=True)

WINDOW_SIZE = 30
STRIDE = 15

def fragment_chain(pdb_path, output_dir, label_prefix):
    st = gemmi.read_structure(pdb_path)
    chain = st[0][0]
    residues = [res for res in chain if res.find_atom("CA", '\0') is not None]
    if len(residues) < WINDOW_SIZE:
        frag_path = os.path.join(output_dir, f"{label_prefix}_full.pdb")
        st.write_pdb(frag_path)
        return [{"path": frag_path, "start_res": 0, "end_res": len(residues), "label": f"{label_prefix}_full"}]
    fragments = []
    for start_idx in range(0, len(residues) - WINDOW_SIZE + 1, STRIDE):
        end_idx = start_idx + WINDOW_SIZE
        new_st = gemmi.Structure()
        new_st.cell = st.cell
        new_model = gemmi.Model("1")
        new_chain = gemmi.Chain(chain.name)
        for res in residues[start_idx:end_idx]:
            new_chain.add_residue(res)
        new_model.add_chain(new_chain)
        new_st.add_model(new_model)
        frag_label = f"{label_prefix}_{start_idx:04d}_{end_idx:04d}"
        frag_path = os.path.join(output_dir, f"{frag_label}.pdb")
        new_st.write_pdb(frag_path)
        fragments.append({"path": frag_path, "start_res": start_idx, "end_res": end_idx, "label": frag_label})
    return fragments

print("Fragmenting spike chains...")
spike_fragment_map = {}
total_spike_frags = 0
for state, pdb_path in spike_chain_files.items():
    frags = fragment_chain(pdb_path, SPIKE_FRAG_DIR, f"spike_{state}")
    spike_fragment_map[state] = frags
    total_spike_frags += len(frags)
    print(f"  spike {state}: {len(frags)} fragments")

print("\nFragmenting antigen chains...")
antigen_fragment_map = {}
total_antigen_frags = 0
for symbol, pdb_path in antigen_chain_files.items():
    frags = fragment_chain(pdb_path, ANTIGEN_FRAG_DIR, symbol)
    antigen_fragment_map[symbol] = frags
    total_antigen_frags += len(frags)
    print(f"  {symbol}: {len(frags)} fragments")

print(f"\nTotal: {total_spike_frags} spike x {total_antigen_frags} antigen fragments")

# Run Foldseek fragment search
print("\nRunning fragment-vs-fragment Foldseek search...")
frag_out_file = os.path.join(FRAG_RESULTS_DIR, "fragment_hits.m8")
FRAG_OUTPUT_COLS = "query,target,alntmscore,fident,alnlen,qlen,tlen,evalue,lddt"
cmd = [FOLDSEEK_BIN, "easy-search", SPIKE_FRAG_DIR, ANTIGEN_FRAG_DIR, frag_out_file,
       FRAG_TMP_DIR, "--format-output", FRAG_OUTPUT_COLS, "--exhaustive-search", "1", "-e", "10"]
subprocess.run(cmd, capture_output=True, text=True)

fragment_hits = []
if os.path.exists(frag_out_file) and os.path.getsize(frag_out_file) > 0:
    with open(frag_out_file) as f:
        for line in f:
            fields = line.strip().split("\t")
            if len(fields) >= 9:
                query_name = fields[0]
                target_name = fields[1]
                spike_state = None
                for state in spike_fragment_map:
                    if f"spike_{state}" in query_name:
                        spike_state = state
                        break
                antigen = None
                for symbol in antigen_fragment_map:
                    if target_name.startswith(symbol):
                        antigen = symbol
                        break
                fragment_hits.append({
                    "query_fragment": query_name, "target_fragment": target_name,
                    "spike_state": spike_state, "antigen": antigen,
                    "tm_score": float(fields[2]), "seq_identity": float(fields[3]),
                    "aln_length": int(fields[4]), "query_length": int(fields[5]),
                    "target_length": int(fields[6]), "evalue": float(fields[7]),
                    "lddt": float(fields[8])})

print(f"{len(fragment_hits)} raw fragment hits.")

# Remove artifacts: TM > 1.0 (overflow) and alnlen < 15 (trivial matches)
MIN_ALN_LENGTH = 15
PATCH_TM_THRESHOLD = 0.3

n_overflow = sum(1 for h in fragment_hits if h["tm_score"] > 1.0)
valid_hits = [h for h in fragment_hits if h["tm_score"] <= 1.0 and h["aln_length"] >= MIN_ALN_LENGTH]
significant_valid = [h for h in valid_hits if h["tm_score"] >= PATCH_TM_THRESHOLD]
significant_valid.sort(key=lambda h: h["tm_score"], reverse=True)

print(f"Overflow artifacts removed: {n_overflow}")
print(f"Valid hits after filtering: {len(valid_hits)}")
print(f"Valid hits with TM >= {PATCH_TM_THRESHOLD}: {len(significant_valid)}")

if significant_valid:
    print(f"\nTop 10 valid fragment hits:")
    for h in significant_valid[:10]:
        print(f"  TM={h['tm_score']:.3f}  alnlen={h['aln_length']}  {h['spike_state']}  "
              f"{h['antigen']}  {h['query_fragment']}  {h['target_fragment']}")

all_results_fragment = {
    "method": "fragment-based Foldseek search",
    "window_size_residues": WINDOW_SIZE, "stride_residues": STRIDE,
    "n_spike_fragments": total_spike_frags, "n_antigen_fragments": total_antigen_frags,
    "n_total_raw_hits": len(fragment_hits), "n_overflow_artifacts_removed": n_overflow,
    "min_alignment_length_filter": MIN_ALN_LENGTH,
    "n_valid_hits_after_filter": len(valid_hits),
    "tm_threshold_for_significance": PATCH_TM_THRESHOLD,
    "n_significant_valid_patches": len(significant_valid),
    "significant_patches": significant_valid[:50],
    "artifact_note": f"Removed {n_overflow} hits with TM>1.0 (overflow) and alnlen<{MIN_ALN_LENGTH}.",
}

## Write `alignment_results.json`

In [ ]:
from datetime import datetime, timezone

alignment_output = {
    "pipeline_step": 4,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab",
    "foldseek_version": subprocess.run([FOLDSEEK_BIN, "version"], capture_output=True, text=True).stdout.strip(),
    "output_columns": OUTPUT_COLS,
    "comparison_set": {
        "spike_states": list(spike_chain_files.keys()),
        "cardiac_antigens": list(antigen_chain_files.keys()),
        "n_pairwise": len(all_results),
    },
    "results": all_results,
    "fragment_analysis": all_results_fragment,
}

output_path = os.path.join(STEP4_DIR, "alignment_results.json")
with open(output_path, "w") as f:
    json.dump(alignment_output, f, indent=2)

print("alignment_results.json written to:", output_path)

tpm1_hits = [r for r in all_results if r["antigen"] == "TPM1" and r["tm_score"] is not None]
if tpm1_hits:
    best = max(tpm1_hits, key=lambda r: r["tm_score"])
    worst = min(tpm1_hits, key=lambda r: r["tm_score"])
    print(f"\nTPM1: highest TM={best['tm_score']:.4f} ({best['spike_state']}), "
          f"lowest TM={worst['tm_score']:.4f} ({worst['spike_state']}), "
          f"delta={best['tm_score']-worst['tm_score']:.4f}")